## 1) Setup

In [ ]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Ensure we can import the helper functions from the app
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'app'))

from utils import load_actuals_jan2024, load_forecasts_jan2024, align_actuals_and_forecasts

# Display settings
pd.options.display.max_rows = 25
pd.options.display.max_columns = 50

print('pandas', pd.__version__, 'numpy', np.__version__)


## 2) Load data (January 2024)

The BMRS API typically requires an API key stored in the `BMRS_API_KEY` environment variable. If you do not have a key, you can obtain one from ELEXON BMRS (https://www.bmreports.com).

In [ ]:
# Ensure the API key is available
if 'BMRS_API_KEY' not in os.environ:
    raise RuntimeError('BMRS_API_KEY is not set in the environment.
Please set it before running this notebook.')

actual = load_actuals_jan2024()
forecasts = load_forecasts_jan2024()

print('Actuals:', actual.shape)
print('Forecasts:', forecasts.shape)

actual.head()

## 3) Forecast error metrics by horizon

We compute accuracy statistics (MAE, RMSE, quantiles) for forecast horizons from 0 to 48 hours. For each horizon, we select the **latest forecast available at least that many hours before the target time**.

In [ ]:
def compute_error_stats(actual_df, forecast_df, horizon_hours, start='2024-01-01', end='2024-01-31'):
    df = align_actuals_and_forecasts(actual_df, forecast_df, horizon_hours, start, end)
    df = df.dropna(subset=['forecast', 'actual']).copy()
    df['error'] = df['forecast'] - df['actual']
    df['abs_error'] = df['error'].abs()

    return {
        'horizon': horizon_hours,
        'count': len(df),
        'mae': df['abs_error'].mean(),
        'rmse': np.sqrt((df['error'] ** 2).mean()),
        'p50': df['abs_error'].quantile(0.5),
        'p90': df['abs_error'].quantile(0.9),
        'p95': df['abs_error'].quantile(0.95),
    }

horizons = list(range(0, 49))
stats = [compute_error_stats(actual, forecasts, h) for h in horizons]
stats_df = pd.DataFrame(stats)
stats_df.head()

### 3.1 Error metrics vs horizon

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(stats_df['horizon'], stats_df['mae'], label='MAE')
ax.plot(stats_df['horizon'], stats_df['rmse'], label='RMSE')
ax.set_xlabel('Forecast horizon (hours)')
ax.set_ylabel('MW')
ax.set_title('Forecast error vs horizon (Jan 2024)')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 4) Error distribution by time of day

In [ ]:
# Pick a representative horizon (e.g., 4 hours)
horizon = 4
df = align_actuals_and_forecasts(actual, forecasts, horizon, '2024-01-01', '2024-01-31')
df = df.dropna(subset=['forecast', 'actual']).copy()
df['error'] = df['forecast'] - df['actual']
df['abs_error'] = df['error'].abs()
df['hour'] = df['startTime'].dt.hour

hour_summary = df.groupby('hour')['abs_error'].agg(['mean', 'median', 'quantile']).reset_index()
# quantile returns 0.5 by default, so we'll calculate 90th pct manually
hour_summary['p90'] = df.groupby('hour')['abs_error'].quantile(0.9).values
hour_summary.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hour_summary['hour'], hour_summary['mean'], label='Mean abs error')
ax.plot(hour_summary['hour'], hour_summary['p90'], label='90th pct abs error')
ax.set_xlabel('Hour of day (UTC)')
ax.set_ylabel('MW')
ax.set_title(f'Error statistics by hour (horizon={horizon}h)')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 5) Actual generation distribution and reliability recommendation

We examine the distribution of actual wind generation in January 2024 and estimate a reliable capacity that is likely to be available a high percentage of the time.

In [ ]:
# Distribution of actual generation
gen = actual['generation'].dropna()
quantiles = gen.quantile([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9]).to_frame('MW')
quantiles

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(gen, bins=60, kde=False, ax=ax)
ax.set_title('Distribution of actual wind generation (Jan 2024)')
ax.set_xlabel('Generation (MW)')
ax.set_ylabel('Count of 30-min intervals')
plt.show()

### Recommendation

Based on the historical distribution, a conservative recommendation is to plan for a capacity that is exceeded in a high percentage of 30-minute intervals (e.g., 90% or 95%).

For January 2024, the 90th percentile of actual generation (i.e., 90% of the time generation is at or below this value) will be used as a proxy for reliable available capacity.

(Adjust the target percentile depending on reliability requirements.)

In [ ]:
reliable_pct = 0.9
reliable_capacity = gen.quantile(reliable_pct)
print(f'Estimated reliable capacity at {int(reliable_pct*100)}th percentile: {reliable_capacity:.0f} MW')

# Provide a small table of candidate reliability levels
for p in [0.75, 0.9, 0.95]:
    print(f'{int(p*100)}th percentile -> {gen.quantile(p):.0f} MW')